# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rattan-Kumar/flyrank-ML-Intership/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Capstone — Refresh / Content Opportunity Scoring

The goal of this capstone is to build a **ranked review queue for content pages**, helping the SEO team decide which pages should be reviewed first. The focus is on identifying potential content opportunities, not making claims about how Google’s algorithm works.

**Lane:** Refresh / Content Opportunity Scoring
**Data:** FlyRank/internship-warehouse (Hugging Face, gated) — `fact_content_daily_performance`, `dim_clients`, and `dim_content`
**Output:** A ranked and reason-coded action queue that helps humans prioritize pages for review.

In this notebook, I query the warehouse directly using **DuckDB over `hf://` Parquet files**, so the full dataset does not need to be downloaded. I create historical features in a leakage-safe way, define a future-looking opportunity label based on the distribution of the available data, and train a machine learning model against a transparent rule-based baseline using the same time-aware data split.

The final output is a ranked list of content pages along with **reason codes** explaining why each page was prioritized. This makes the recommendations easier for the SEO team to understand and review.

The notebook is designed to run from top to bottom in **Google Colab**, with the required internship-token secret configured. Heavy DuckDB scans are performed once and cached in `work/outputs/`, making subsequent runs much faster.


In [1]:
%pip install -q duckdb pandas numpy scikit-learn matplotlib huggingface_hub

In [3]:
# Authenticate — token stays in Colab Secrets, never in a cell or printed.
from google.colab import userdata
import os

HF_TOKEN = userdata.get("intership")
os.environ["HF_TOKEN"] = HF_TOKEN  # lets duckdb's httpfs pick it up too
print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [4]:
import duckdb

con = duckdb.connect()
con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{}'
    )
""".format(HF_TOKEN))

# Query hf:// paths directly — no snapshot_download, no full warehouse copy.
REPO = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"read_parquet('{REPO}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REPO}/dim_content.parquet')"

# Near-free sanity check: metadata only, not a full scan.
counts = con.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM {DIM_CLIENTS})  AS n_clients,
        (SELECT COUNT(*) FROM {DIM_CONTENT})  AS n_content
""").df()
counts


,n_clients,n_content
0,104,519606


## 1. Question

*The research question and the decision it supports.*

### 1. Research Question

The main question is: **Can we use a content page’s search and engagement history up to a specific cutoff date to rank pages that are most likely to experience a meaningful decline in future search demand?** The goal is to create a review queue that performs better than a simple momentum-based rule.

This supports a practical SEO decision. Since an editorial or SEO reviewer has limited time and cannot manually check every page each month, the queue helps them decide **which pages to review first and why**. The recommendations are meant to support human decision-making—they do not guarantee that a refresh will work or explain how Google’s algorithm behaves.

### What the Output Provides

The final output is a ranked list of anonymized **`content_hash_id`** values, each with a score, an action, and reason codes. Possible actions include **`REFRESH_REVIEW`**, **`MONITOR`**, **`PROTECT`**, **`INVESTIGATE`**, and **`RECOVERY_REVIEW`**. The reason codes are based on the same features used by the model, making the recommendations easier for a reviewer to understand.

### What This Does Not Claim

This project does **not** attempt to predict search-engine ranking behavior, and it does not claim that refreshing a page will cause its traffic to recover. The term **“decline”** is simply an operational label defined from the page’s observed historical performance. It should not be interpreted as a direct measure of content quality.


In [5]:
# No computation needed for this section — the question is fixed above.
# (Kept as an empty-but-present code cell so the notebook structure matches the assignment skeleton.)
print("Research question set. Proceeding to Data.")

Research question set. Proceeding to Data.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

Here’s a clearer, more humanized version while keeping the methodology precise:

### 1. Research Question

The main question is whether we can use a content page’s **search and engagement history up to a specific cutoff date** to identify and rank the pages that are most likely to experience a meaningful decline in future search demand. The goal is to create a practical review queue that can perform better than a simple momentum-based rule.

The problem is based on a real-world SEO workflow. An editorial or SEO reviewer has limited time and cannot manually check every content page each month. The ranking system helps them decide **which pages to review first and why**, using clear reason codes to support each recommendation. It is designed as decision-support for a human reviewer, not as an automated decision-maker.

### What the Output Provides

The final output is a ranked list of anonymized **`content_hash_id`** values. Each page receives a score, a recommended action, and reason codes based on the same features used by the model.

The possible actions are:

* **`REFRESH_REVIEW`** — the page may be worth reviewing for a content refresh.
* **`MONITOR`** — the page should be watched for further changes.
* **`PROTECT`** — the page is performing well and may need protection from unnecessary changes.
* **`INVESTIGATE`** — the signals are unusual and require further human review.
* **`RECOVERY_REVIEW`** — the page shows signs that a previous decline may be worth investigating.

### What This Does Not Claim

This project does **not** attempt to predict Google or search-engine ranking behavior. It also does not claim that refreshing a page will automatically recover its traffic.

The term **“decline”** is an operational label created from the observed historical data using the defined methodology. It should not be interpreted as a measure of content quality.

Overall, the system is intended to answer a simple practical question: **“Which pages should a human SEO reviewer look at first, and what signals explain that recommendation?”**


In [6]:
# --- Metadata-only checks: near-free, confirm we're pointed at the right table/date range ---
FACT_ALL = f"read_parquet('{REPO}/fact_content_daily_performance/*/*.parquet')"

meta = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {FACT_ALL}
""").df()
meta


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,min_date,max_date
0,78835655,2025-01-27,2026-06-30


In [7]:
# --- Grain probe on ONE mid-panel month first (cheap) — confirm duplicate page-days exist,
# then confirm they are EXACT duplicates before deciding to dedupe.
FACT_PROBE_MONTH = f"read_parquet('{REPO}/fact_content_daily_performance/month=2026-01/*.parquet')"

dupe_probe = con.sql(f"""
    SELECT content_hash_id, report_date, COUNT(*) AS n
    FROM {FACT_PROBE_MONTH}
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
dupe_probe

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,report_date,n


In [8]:
# For one duplicated (content_hash_id, report_date) pair, confirm every column has nunique()==1
# i.e. the "duplicate" rows are exact copies, not distinct observations.
if len(dupe_probe):
    cid, rdate = dupe_probe.iloc[0]["content_hash_id"], dupe_probe.iloc[0]["report_date"]
    dup_rows = con.sql(f"""
        SELECT * FROM {FACT_PROBE_MONTH}
        WHERE content_hash_id = '{cid}' AND report_date = DATE '{rdate}'
    """).df()
    print("Rows found for this page-day:", len(dup_rows))
    print("Columns where the duplicate rows actually differ (should be EMPTY):")
    print([c for c in dup_rows.columns if dup_rows[c].nunique(dropna=False) > 1])
else:
    print("No duplicates found in this month's probe — will still dedupe defensively downstream.")


No duplicates found in this month's probe — will still dedupe defensively downstream.


In [9]:
# --- Per-client history coverage: define windows relative to each client's OWN start date,
# never a single global calendar window (unbalanced panel warning from the data skill).
clients = con.sql(f"""
    SELECT client_hash_id, gsc_data_start, ga4_data_start
    FROM {DIM_CLIENTS}
""").df()
clients.describe(include="all")

,client_hash_id,gsc_data_start,ga4_data_start
count,104,67,51
unique,104,NaN,NaN
top,client_04660893ae39614a,NaN,NaN
freq,1,NaN,NaN
mean,NaN,2025-11-17 00:42:59.104477,2026-02-23 05:38:49.411764
min,NaN,2025-01-27 00:00:00,2025-10-29 00:00:00
25%,NaN,2025-09-24 00:00:00,2026-02-19 00:00:00
50%,NaN,2025-11-05 00:00:00,2026-02-20 00:00:00
75%,NaN,2026-02-19 00:00:00,2026-03-21 12:00:00
max,NaN,2026-06-02 00:00:00,2026-06-01 00:00:00


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

Here’s a clearer, more humanized version that keeps the methodology and technical details accurate:

### Unit of Analysis

The raw data is recorded at the **content page × day** level. However, the actual modeling unit is **content page × cutoff date**. For each chosen cutoff date **T**, the model uses only information available on or before T to create the historical features, while the label is calculated only from data after T.

Using multiple cutoff dates for the same page creates additional training examples without inventing any data. Since each feature window and label window are separated in time, the same page appearing at different cutoffs is not considered leakage. This is similar to a repeated-observation or rolling-origin setup commonly used for forecasting.

### Cutoff Dates

| Cutoff (T) | Feature Window                    | Label Window                      | Role  |
| ---------- | --------------------------------- | --------------------------------- | ----- |
| 2025-09-30 | 2025-08-05 → 2025-09-30 (56 days) | 2025-10-01 → 2025-10-28 (28 days) | Train |
| 2025-11-30 | 2025-10-06 → 2025-11-30 (56 days) | 2025-12-01 → 2025-12-28 (28 days) | Train |
| 2026-01-31 | 2025-12-07 → 2026-01-31 (56 days) | 2026-02-01 → 2026-02-28 (28 days) | Test  |

All three cutoff and label periods end well before June 2026, which is the sealed sample month. They also occur before the fixed 90-day query window, so the data used for modeling does not overlap with the sealed test period.

The 56-day feature window is split into two 28-day periods: **recent_28** and **prior_28**. This allows us to measure changes in performance while keeping the amount of required history manageable for clients with shorter tracking periods.

### Eligibility

A **page × cutoff** observation is included only when the required historical information is available. Specifically:

* The client’s `gsc_data_start` must be at least 55 days before the cutoff, ensuring the full feature window exists.
* `gsc_data_available IS TRUE` must hold for the days being used. This is intentional because the availability field can contain NULL values, and using `IS TRUE` avoids incorrectly treating missing information as available.
* The page must have reached a minimum level of visibility during `recent_28`, based on an impressions threshold derived from the data. A page with virtually no impressions cannot meaningfully be described as declining in search visibility.

### Features

For both **recent_28** and **prior_28**, I calculate totals for:

* GSC impressions
* GSC clicks
* Organic sessions
* GA4 pageviews
* GA4 engaged sessions
* Total engagement time
* Scroll events

I also calculate the average daily **GSC position**.

From these values, the model features include changes in:

* Clicks
* Impressions
* Organic sessions
* Search position
* Engagement rate

I also include **click volatility**, measured using the standard deviation of daily clicks across the 56-day feature window.

Finally, I calculate recent-to-prior ratios for clicks and impressions. A small smoothing value is added to avoid division-by-zero problems and extreme ratios for pages with very little activity.

### Future Opportunity Label

The label is created using the **28-day period after the cutoff date**, so it contains information that would not have been available when making the original prediction.

The main measure is the future-to-recent click ratio:

**`future_click_ratio = (future_clicks + 1) / (recent_clicks + 1)`**

Rather than choosing an arbitrary decline threshold such as 20% or 30%, I examine the distribution of this ratio among eligible pages and use its **lower quartile** as the threshold. A page receives a positive label when its future click ratio falls at or below this data-driven threshold and it also meets the required visibility floor.

This makes the definition of “meaningful decline” dependent on the actual distribution of the dataset rather than an assumption chosen in advance.

### Key Assumption

The approach assumes that a decline in a page’s click ratio can serve as a reasonable proxy for **losing search relevance or demand**. However, this is not a direct measure of content quality. Search performance can change for many reasons that are not visible in the dataset, such as seasonality, competitor activity, or changes in search-result features.

Using time-based cutoffs instead of a random train/test split makes the experiment more realistic. It better reflects the way the system would actually be used: **use information available today to score pages, then observe what happens afterward.**


In [10]:
# --- Build the month path LIST covering all cutoffs' feature + label windows (hf:// needs an
# explicit list, not a brace-glob).
MONTHS_NEEDED = ["2025-08","2025-09","2025-10","2025-11","2025-12","2026-01","2026-02"]
month_paths = [f"{REPO}/fact_content_daily_performance/month={m}/*.parquet" for m in MONTHS_NEEDED]
month_list_sql = ", ".join(f"'{p}'" for p in month_paths)

FACT_WINDOW = f"read_parquet([{month_list_sql}])"

# Grain-safe base: dedupe exact-duplicate page-days once, up front, in SQL.
BASE_CTE = f"""
base AS (
    SELECT
        content_hash_id,
        client_hash_id,
        report_date,
        ANY_VALUE(gsc_impressions)          AS gsc_impressions,
        ANY_VALUE(gsc_clicks)               AS gsc_clicks,
        ANY_VALUE(gsc_avg_position)         AS gsc_avg_position,
        ANY_VALUE(sessions_organic)         AS sessions_organic,
        ANY_VALUE(ga4_pageviews)            AS ga4_pageviews,
        ANY_VALUE(ga4_engaged_sessions)     AS ga4_engaged_sessions,
        ANY_VALUE(ga4_total_engagement_sec) AS ga4_total_engagement_sec,
        ANY_VALUE(scroll_events)            AS scroll_events,
        ANY_VALUE(gsc_data_available)       AS gsc_data_available,
        ANY_VALUE(ga4_data_available)       AS ga4_data_available
    FROM {FACT_WINDOW}
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id, report_date
)
"""
print("Base CTE ready.")

Base CTE ready.


In [11]:
def cutoff_query(cutoff, feat_start, prior_split, label_end):
    """Build the per-cutoff aggregation SQL. All dates are strings 'YYYY-MM-DD'.
    recent_28  = (prior_split, cutoff]
    prior_28   = [feat_start, prior_split]
    future_28  = (cutoff, label_end]
    """
    return f"""
    WITH {BASE_CTE}
    , eligible_clients AS (
        SELECT client_hash_id
        FROM {DIM_CLIENTS}
        WHERE gsc_data_start <= DATE '{feat_start}'
    )
    SELECT
        b.content_hash_id,
        DATE '{cutoff}' AS cutoff_date,

        SUM(CASE WHEN b.report_date > DATE '{prior_split}' AND b.report_date <= DATE '{cutoff}' THEN b.gsc_impressions END)          AS recent_impressions,
        SUM(CASE WHEN b.report_date > DATE '{prior_split}' AND b.report_date <= DATE '{cutoff}' THEN b.gsc_clicks END)               AS recent_clicks,
        AVG(CASE WHEN b.report_date > DATE '{prior_split}' AND b.report_date <= DATE '{cutoff}' THEN b.gsc_avg_position END)          AS recent_position,
        SUM(CASE WHEN b.report_date > DATE '{prior_split}' AND b.report_date <= DATE '{cutoff}' THEN b.sessions_organic END)          AS recent_organic_sessions,
        SUM(CASE WHEN b.report_date > DATE '{prior_split}' AND b.report_date <= DATE '{cutoff}' THEN b.ga4_engaged_sessions END)      AS recent_engaged_sessions,
        SUM(CASE WHEN b.report_date > DATE '{prior_split}' AND b.report_date <= DATE '{cutoff}' THEN b.ga4_total_engagement_sec END)  AS recent_engagement_sec,
        SUM(CASE WHEN b.report_date > DATE '{prior_split}' AND b.report_date <= DATE '{cutoff}' THEN b.scroll_events END)             AS recent_scroll_events,

        SUM(CASE WHEN b.report_date > DATE '{feat_start}' AND b.report_date <= DATE '{prior_split}' THEN b.gsc_impressions END)         AS prior_impressions,
        SUM(CASE WHEN b.report_date > DATE '{feat_start}' AND b.report_date <= DATE '{prior_split}' THEN b.gsc_clicks END)              AS prior_clicks,
        AVG(CASE WHEN b.report_date > DATE '{feat_start}' AND b.report_date <= DATE '{prior_split}' THEN b.gsc_avg_position END)         AS prior_position,
        SUM(CASE WHEN b.report_date > DATE '{feat_start}' AND b.report_date <= DATE '{prior_split}' THEN b.sessions_organic END)         AS prior_organic_sessions,
        SUM(CASE WHEN b.report_date > DATE '{feat_start}' AND b.report_date <= DATE '{prior_split}' THEN b.ga4_engaged_sessions END)     AS prior_engaged_sessions,

        STDDEV_SAMP(CASE WHEN b.report_date > DATE '{feat_start}' AND b.report_date <= DATE '{cutoff}' THEN b.gsc_clicks END)         AS click_volatility_56d,

        SUM(CASE WHEN b.report_date > DATE '{cutoff}' AND b.report_date <= DATE '{label_end}' THEN b.gsc_clicks END)                  AS future_clicks,
        SUM(CASE WHEN b.report_date > DATE '{cutoff}' AND b.report_date <= DATE '{label_end}' THEN b.gsc_impressions END)             AS future_impressions,
        SUM(CASE WHEN b.report_date > DATE '{cutoff}' AND b.report_date <= DATE '{label_end}' THEN b.sessions_organic END)            AS future_organic_sessions

    FROM base b
    JOIN eligible_clients ec ON ec.client_hash_id = b.client_hash_id
    WHERE b.report_date > DATE '{feat_start}' AND b.report_date <= DATE '{label_end}'
    GROUP BY b.content_hash_id
    """

CUTOFF_SPECS = [
    dict(cutoff="2025-09-30", feat_start="2025-08-05", prior_split="2025-09-02", label_end="2025-10-28", split="train"),
    dict(cutoff="2025-11-30", feat_start="2025-10-06", prior_split="2025-11-02", label_end="2025-12-28", split="train"),
    dict(cutoff="2026-01-31", feat_start="2025-12-07", prior_split="2026-01-03", label_end="2026-02-28", split="test"),
]
for spec in CUTOFF_SPECS:
    print(spec)

{'cutoff': '2025-09-30', 'feat_start': '2025-08-05', 'prior_split': '2025-09-02', 'label_end': '2025-10-28', 'split': 'train'}
{'cutoff': '2025-11-30', 'feat_start': '2025-10-06', 'prior_split': '2025-11-02', 'label_end': '2025-12-28', 'split': 'train'}
{'cutoff': '2026-01-31', 'feat_start': '2025-12-07', 'prior_split': '2026-01-03', 'label_end': '2026-02-28', 'split': 'test'}


In [12]:
# --- Run the full scan ONCE per cutoff, cache to disk so reruns don't re-hit the warehouse.
import os
import pandas as pd
os.makedirs("work/outputs", exist_ok=True)
CACHE_PATH = "work/outputs/capstone_features_raw.parquet"

if os.path.exists(CACHE_PATH):
    print("Loading cached features — delete the file to force a fresh scan.")
    raw = pd.read_parquet(CACHE_PATH)
else:
    frames = []
    for spec in CUTOFF_SPECS:
        q = cutoff_query(spec["cutoff"], spec["feat_start"], spec["prior_split"], spec["label_end"])
        df = con.sql(q).df()
        df["split"] = spec["split"]
        frames.append(df)
        print(spec["cutoff"], "->", len(df), "content rows")
    raw = pd.concat(frames, ignore_index=True)
    raw.to_parquet(CACHE_PATH)

raw.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2025-09-30 -> 53683 content rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2025-11-30 -> 84676 content rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2026-01-31 -> 146388 content rows


(284747, 19)

In [13]:
# --- Derived features (all built from recent_28 / prior_28 only — no future_* columns here) ---
feat = raw.copy()

# Smoothing constant avoids divide-by-zero / infinite ratios for idle pages.
K = 1.0

feat["click_change"]              = feat["recent_clicks"] - feat["prior_clicks"]
feat["impression_change"]         = feat["recent_impressions"] - feat["prior_impressions"]
feat["organic_session_change"]    = feat["recent_organic_sessions"] - feat["prior_organic_sessions"]
feat["position_change"]           = feat["recent_position"] - feat["prior_position"]  # + = worse
feat["recent_engagement_rate"]    = feat["recent_engaged_sessions"] / (feat["recent_organic_sessions"] + K)
feat["prior_engagement_rate"]     = feat["prior_engaged_sessions"]  / (feat["prior_organic_sessions"]  + K)
feat["engagement_rate_change"]    = feat["recent_engagement_rate"] - feat["prior_engagement_rate"]
feat["click_ratio_recent_prior"]  = (feat["recent_clicks"] + K) / (feat["prior_clicks"] + K)
feat["impression_ratio_recent_prior"] = (feat["recent_impressions"] + K) / (feat["prior_impressions"] + K)

FEATURE_COLS = [
    "recent_impressions", "recent_clicks", "recent_position", "recent_organic_sessions",

    "recent_engagement_rate", "recent_scroll_events",
    "click_change", "impression_change", "organic_session_change", "position_change",
    "engagement_rate_change", "click_ratio_recent_prior", "impression_ratio_recent_prior",
    "click_volatility_56d",
]

feat[FEATURE_COLS].describe().T

,count,mean,std,min,25%,50%,75%,max
recent_impressions,232645.0,987.487528,3699.535364,1.000000,13.000000,107.000000,586.000000,319004.000000
recent_clicks,232645.0,3.683238,20.916658,0.000000,0.000000,0.000000,1.000000,3232.000000
recent_position,232645.0,15.984159,18.128553,0.000000,5.000000,8.672986,19.311155,354.000000
recent_organic_sessions,153730.0,2.954108,25.575857,0.000000,0.000000,0.000000,0.000000,2534.000000
recent_engagement_rate,153730.0,0.010671,0.083632,0.000000,0.000000,0.000000,0.000000,6.000000
recent_scroll_events,153730.0,0.360242,3.343320,0.000000,0.000000,0.000000,0.000000,507.000000
click_change,188099.0,0.855528,13.104015,-1346.000000,0.000000,0.000000,1.000000,1969.000000
impression_change,188099.0,205.869856,2632.386507,-157461.000000,-10.000000,13.000000,160.000000,293549.000000
organic_session_change,128332.0,1.625604,22.839603,-1698.000000,0.000000,0.000000,0.000000,2403.000000
position_change,188099.0,-1.104485,13.337239,-406.000000,-3.479599,-0.125189,2.531086,285.250000


In [14]:
# --- Visibility floor, read off the data (not assumed): use the median of recent_impressions
# among pages with ANY recent visibility as the eligibility floor. A page below this line wasn't
# meaningfully "visible" in the recent window, so a future click drop there isn't an opportunity —
# it's an absent page.
visible_mask = feat["recent_impressions"] > 0
VISIBILITY_FLOOR = feat.loc[visible_mask, "recent_impressions"].median()
print("Visibility floor (median recent_impressions among visible pages):", VISIBILITY_FLOOR)

eligible = feat[feat["recent_impressions"] >= VISIBILITY_FLOOR].copy()
print("Eligible (page, cutoff) rows:", len(eligible), "of", len(feat), "total")
eligible["split"].value_counts()


Visibility floor (median recent_impressions among visible pages): 107.0
Eligible (page, cutoff) rows: 116570 of 284747 total


,count
split,
test,61414
train,55156


In [16]:
# --- Label: future_click_ratio, threshold read from the ELIGIBLE, TRAIN-SPLIT distribution only
# (never from test, to avoid leaking test-set shape into the label's own definition).
eligible["future_click_ratio"] = (eligible["future_clicks"].fillna(0) + K) / (eligible["recent_clicks"] + K)

train_mask = eligible["split"] == "train"
DECLINE_THRESHOLD = eligible.loc[train_mask, "future_click_ratio"].quantile(0.25)
print("Decline threshold (25th pct of future_click_ratio, TRAIN only):", round(DECLINE_THRESHOLD, 3))

eligible["opportunity_label"] = (eligible["future_click_ratio"] <= DECLINE_THRESHOLD).astype(int)

print("\nLabel prevalence by split:")
print(eligible.groupby("split")["opportunity_label"].agg(["mean", "sum", "count"]))

Decline threshold (25th pct of future_click_ratio, TRAIN only): 0.667

Label prevalence by split:
           mean    sum  count
split                        
test   0.181587  11152  61414
train  0.255765  14107  55156


Here’s a more natural and humanized version:

### Class Balance

I checked the class balance directly instead of trying to artificially fix it with techniques such as **SMOTE**. Since the label is defined using the **25th percentile of the training distribution**, having roughly 25% of eligible training rows labeled as positive is expected by design. It does not, by itself, indicate a problem with the dataset.

I am not using SMOTE or similar resampling methods because this is primarily a **ranking problem**, not a traditional classification task where overall accuracy is the main concern. The real question is: **which pages should appear at the top of the review queue?** Therefore, the more relevant metrics are **Precision@K, Recall@K, and Average Precision**, and these should always be interpreted alongside the underlying positive-class base rate.

For the main evaluation, I use **Precision@50** because the eligible test population is large enough to support a meaningful top-50 review queue. This metric directly reflects the practical goal of getting the most relevant pages into the hands of the SEO team first.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

### Baseline Rule

The baseline follows a simple idea: **a page should be reviewed first when its clicks are falling and its search position is getting worse compared with its own recent performance.** In other words, we rank pages by their momentum, putting the most concerning ones at the top.

This is implemented using a transparent score with no fitted model weights:

`baseline_score = -click_ratio_recent_prior + position_change`

A higher score indicates more concerning momentum. Because the weights are not learned from the data, the baseline provides a simple and transparent benchmark that the ML model should be able to beat.

### Validation Design

I use a **time-aware train/test split rather than a random split**. The model is trained on the **2025-09-30** and **2025-11-30** cutoff datasets, giving 48,697 eligible rows after removing missing features, with a 25.6% positive rate.

The model is then evaluated on the later **2026-01-31** cutoff, which contains 30,302 eligible rows with a 19.5% positive rate. This ensures that the test period comes strictly after the periods used for training.

For a fair comparison, the baseline is evaluated on exactly the same test rows and against exactly the same labels.

### Leakage Audit

I also checked the pipeline carefully for potential data leakage:

| Check                                                             | Result                                                                                    |
| ----------------------------------------------------------------- | ----------------------------------------------------------------------------------------- |
| Future clicks, impressions, or organic sessions used as features? | **No** — `future_*` columns are used only to create the label.                            |
| Future labels included in training features?                      | **No** — the label is calculated separately from the future outcome window.               |
| Label threshold calculated using the full dataset?                | **No** — the threshold of **0.667** is calculated using the training split only.          |
| Scaling or encoding fitted using test data?                       | **No** — preprocessing is fitted only on the training data.                               |
| Do features contain information after the cutoff?                 | **No** — features come only from `recent_*`, `prior_*`, or changes between those periods. |
| Is the train/test split time-based?                               | **Yes** — training cutoffs occur strictly before the test cutoff.                         |
| Does population selection use future outcome information?         | **No** — eligibility is based only on the feature window and dimension data.              |

### Results

| Method           | Precision@50 | Hits in Top 50 | Recall@50 | Average Precision | Base Rate |
| ---------------- | -----------: | -------------: | --------: | ----------------: | --------: |
| Baseline (rules) |        0.120 |              6 |     0.001 |             0.188 |     0.195 |
| Random Forest    |        0.180 |              9 |     0.002 |             0.374 |     0.195 |

These results need to be interpreted carefully. At the strict **top-50** level, neither approach provides strong evidence of beating random selection. With a 19.5% base rate, randomly selecting 50 pages would be expected to contain about **9–10 positive pages** (0.195 × 50 ≈ 9.75). The Random Forest found 9, which is essentially around the level expected by chance, while the baseline found only 6.

Therefore, **Precision@50 does not support a strong claim that the ML model wins**.

There is, however, a more encouraging signal in **Average Precision**. The Random Forest achieved **0.374**, compared with **0.188** for the baseline—approximately twice as high. This suggests that the model provides better ranking quality when considering the entire ranked list, even though that improvement does not translate into a clear advantage within the first 50 pages.

A larger top-K evaluation might reveal whether this broader ranking advantage becomes more useful in practice, but those additional metrics were not calculated in this run. They should not be invented or reported without actually computing them.

The most honest conclusion is therefore that the model shows **some useful ranking signal overall, but its advantage at the very top of the queue is weak**.

### Feature Importance

The feature importance results also provide a useful sanity check:

| Feature                         | Importance |
| ------------------------------- | ---------: |
| `recent_clicks`                 |      0.377 |
| `click_volatility_56d`          |      0.230 |
| `click_ratio_recent_prior`      |      0.130 |
| `click_change`                  |      0.086 |
| `recent_organic_sessions`       |      0.040 |
| `organic_session_change`        |      0.037 |
| `recent_impressions`            |      0.028 |
| `recent_scroll_events`          |      0.024 |
| `impression_change`             |      0.012 |
| `recent_position`               |      0.011 |
| `recent_engagement_rate`        |      0.009 |
| `impression_ratio_recent_prior` |      0.008 |
| `engagement_rate_change`        |      0.005 |
| `position_change`               |      0.004 |

The most important feature is **`recent_clicks`**, accounting for about **37.7%** of the total importance. This is worth noting, but it does not automatically indicate leakage. The target is based on **future clicks**, while `recent_clicks` comes entirely from the historical feature window.

The relationship is therefore plausible: pages with more recent click activity have more opportunity to experience a measurable change in their future click ratio. Overall, the feature importance results do not show an obvious label-derived or future-information feature dominating the model.


# **Reading the errors**
Of the 30,302 test rows: 363 false negatives (real declines the model missed) and 10,747 false positives (pages flagged that did not decline) at a naive 0.5 probability threshold. The high false-positive count at threshold 0.5 is itself informative: a class_weight="balanced" random forest scored against an already-elevated 19.5% base rate pushes many borderline pages above 0.5. This is a reason to use the ranked score for prioritization (as this project does) rather than the 0.5 threshold as a hard yes/no cutoff — the queue, not the classifier's binary label, is the actual product here.

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
import numpy as np

train_df = eligible[eligible["split"] == "train"].dropna(subset=FEATURE_COLS).reset_index(drop=True)
test_df  = eligible[eligible["split"] == "test"].dropna(subset=FEATURE_COLS).reset_index(drop=True)

X_train, y_train = train_df[FEATURE_COLS], train_df["opportunity_label"]
X_test,  y_test  = test_df[FEATURE_COLS],  test_df["opportunity_label"]

print("Train rows:", len(train_df), " base rate:", round(y_train.mean(), 3))
print("Test rows: ", len(test_df),  " base rate:", round(y_test.mean(), 3))

Train rows: 48697  base rate: 0.256
Test rows:  30302  base rate: 0.195


In [18]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    top_k = np.asarray(labels)[order[:k]]
    return top_k.mean(), top_k.sum()

def recall_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    top_k = np.asarray(labels)[order[:k]]
    total_positive = np.asarray(labels).sum()
    return (top_k.sum() / total_positive) if total_positive > 0 else np.nan

K_EVAL = 50

# --- Baseline: rule score, no fitting, evaluated on the identical test rows ---
baseline_test_score = -test_df["click_ratio_recent_prior"] + test_df["position_change"].fillna(0)

# --- Model: RandomForest, fit on TRAIN ONLY ---
model = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20,
                                class_weight="balanced", random_state=42)
model.fit(X_train, y_train)
model_test_score = model.predict_proba(X_test)[:, 1]

results = []
for name, scores in [("baseline_rules", baseline_test_score), ("random_forest", model_test_score)]:
    p_at_k, hits = precision_at_k(scores, y_test, K_EVAL)
    r_at_k = recall_at_k(scores, y_test, K_EVAL)
    ap = average_precision_score(y_test, scores)
    results.append(dict(method=name, precision_at_50=round(p_at_k, 3), hits_in_top_50=int(hits),
                         recall_at_50=round(r_at_k, 3), average_precision=round(ap, 3)))

results_table = pd.DataFrame(results)
results_table["base_rate"] = round(y_test.mean(), 3)
results_table

,method,precision_at_50,hits_in_top_50,recall_at_50,average_precision,base_rate
0,baseline_rules,0.12,6,0.001,0.188,0.195
1,random_forest,0.18,9,0.002,0.375,0.195


In [19]:
# --- Save the comparison table as the receipt the paper's numbers trace back to ---
results_table.to_json("work/outputs/capstone_model_comparison.json", orient="records", indent=2)
print("Saved -> work/outputs/capstone_model_comparison.json")
results_table

Saved -> work/outputs/capstone_model_comparison.json


,method,precision_at_50,hits_in_top_50,recall_at_50,average_precision,base_rate
0,baseline_rules,0.12,6,0.001,0.188,0.195
1,random_forest,0.18,9,0.002,0.375,0.195


In [20]:
# --- Feature importance, sanity-checked (not just celebrated) ---
importances = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print("Top features:")
print(importances)
print()
print("Sanity check: no single feature should tower over the rest (a near-total share would suggest")
print("a leaked or label-derived column slipped into FEATURE_COLS). Top feature's share of total importance:",
      round(importances.iloc[0] / importances.sum(), 3))

Top features:
recent_clicks                    0.380076
click_volatility_56d             0.229126
click_ratio_recent_prior         0.128195
click_change                     0.084970
recent_organic_sessions          0.041281
organic_session_change           0.037173
recent_impressions               0.028670
recent_scroll_events             0.022122
impression_change                0.012028
recent_position                  0.011099
recent_engagement_rate           0.009402
impression_ratio_recent_prior    0.007179
engagement_rate_change           0.005046
position_change                  0.003634
dtype: float64

Sanity check: no single feature should tower over the rest (a near-total share would suggest
a leaked or label-derived column slipped into FEATURE_COLS). Top feature's share of total importance: 0.38


In [22]:
# --- Read the errors, don't just trust the score: show 3 concrete cases the model got wrong ---
test_df = test_df.copy()
test_df["model_score"] = model_test_score
test_df["model_pred"]  = (model_test_score >= 0.5).astype(int)

false_negatives = test_df[(test_df["opportunity_label"] == 1) & (test_df["model_pred"] == 0)]
false_positives = test_df[(test_df["opportunity_label"] == 0) & (test_df["model_pred"] == 1)]

print("False negatives (missed real declines):", len(false_negatives))
print("False positives (flagged pages that didn't decline):", len(false_positives))
cols_to_show = ["content_hash_id", "recent_impressions", "recent_clicks", "click_ratio_recent_prior",
                 "position_change", "future_click_ratio", "model_score"]
false_negatives[cols_to_show].head(3)

False negatives (missed real declines): 373
False positives (flagged pages that didn't decline): 10709


,content_hash_id,recent_impressions,recent_clicks,click_ratio_recent_prior,position_change,future_click_ratio,model_score
235,content_00bc0b13cd8c24b1,7867.0,28.0,2.416667,0.804046,0.586207,0.468110
250,content_c933fa654e7e3e5d,8086.0,22.0,2.090909,-0.358136,0.565217,0.468585
368,content_d8de0f6c7379663a,39463.0,202.0,1.253086,-0.106446,0.517241,0.411962


## 5. Limitations

*What this work cannot claim.*

Here’s a polished, humanized version suitable for your capstone report:

### Key Limitations and Honest Interpretation

* This dataset contains **observed historical behavior** from a portfolio of pseudonymized clients. Since it is not a controlled experiment, the results show associations rather than cause-and-effect relationships.

* The overall result is **mixed**, and that is important to report honestly. At the strict top-50 cutoff, the model achieved a Precision@50 of **0.18**, which is only slightly below the 19.5% base rate. The rule-based baseline achieved **0.12**, which is actually below the level expected from random selection. The model’s stronger result is its **Average Precision of 0.374**, compared with 0.188 for the baseline. This suggests better ranking quality across the full list, but it should not be interpreted as proof that the first 50 recommendations are consistently correct.

* The model identifies **patterns and associations** that can help prioritize pages for human review. It does not explain or predict Google’s ranking algorithm, and a high model score does not guarantee that refreshing a page will recover clicks or improve its search position.

* The **`opportunity_label`** is an operational definition based on the 25th-percentile future click ratio, with a threshold of **0.667** calculated from the training data. It is a useful proxy for identifying pages that experienced a meaningful decline in this dataset, but it is not an objective measure of content quality. The threshold could also change with a different portfolio or time period.

* Data coverage is uneven across clients and signals. Only **67 of 104 clients** have a confirmed `gsc_data_start`, while only **51 of 104** have a confirmed `ga4_data_start`. GA4-based features such as `recent_organic_sessions` and engagement rate also have much lower coverage (around 54% of feature rows) compared with search features such as `recent_impressions` (around 82%). The eligibility filter removes observations without sufficient reliable history, meaning those pages are not scored. This is an intentional design choice rather than something hidden in the pipeline.

* The evaluation is based on **one time-aware holdout from one dataset snapshot**. Performance could be different during another quarter, season, or client mix, so the results should not automatically be assumed to generalize.

* The definition of **“meaningful decline”** focuses specifically on search-referred clicks. A page could still be performing well through other channels, such as direct, referral, social, paid traffic, or AI-assistant referrals, while being flagged by this system.

* Using a simple **0.5 probability threshold** produces many more false positives (**10,747**) than false negatives (**363**) on the test set. This is partly a consequence of using `class_weight="balanced"`, which can push the predicted probabilities in a way that is not ideal for a binary yes/no decision. It reinforces the idea that this system should be used primarily as a **ranked prioritization tool**, rather than applying a fixed probability threshold to automatically classify pages.


In [23]:
print("Limitations are documented above in markdown; no computation required for this section.")

Limitations are documented above in markdown; no computation required for this section.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Here’s a more polished and humanized version:

### Reason Codes

To make the recommendations easy to understand, each ranked page is given **deterministic reason codes**. These codes are based on the same `recent_*` and `prior_*` features used by the model, so a reviewer can understand why a page was prioritized without needing to inspect the model itself.

| Reason Code                  | Trigger                                                                                              |
| ---------------------------- | ---------------------------------------------------------------------------------------------------- |
| `DECLINING_CLICKS`           | `click_ratio_recent_prior` falls within its bottom quartile                                          |
| `DECLINING_IMPRESSIONS`      | `impression_ratio_recent_prior` falls within its bottom quartile                                     |
| `POSITION_WORSENING`         | `position_change > 0` beyond a small noise range                                                     |
| `ORGANIC_SESSION_DROP`       | `organic_session_change < 0`                                                                         |
| `ENGAGEMENT_DECLINE`         | `engagement_rate_change < 0`                                                                         |
| `HIGH_VISIBILITY_LOW_CLICKS` | `recent_impressions` is above its median while `recent_clicks` is below its median                   |
| `RECOVERY_SIGNAL`            | `click_ratio_recent_prior` falls within its top quartile, indicating improvement rather than decline |
| `STABLE_MONITOR`             | None of the other reason codes are triggered                                                         |

### Action Playbook

The reason codes are then combined with the model score to suggest a practical action:

| Action            | When It Is Used                                                                                                    |
| ----------------- | ------------------------------------------------------------------------------------------------------------------ |
| `REFRESH_REVIEW`  | The model score is high and `DECLINING_CLICKS` or `POSITION_WORSENING` is present                                  |
| `INVESTIGATE`     | The model score is high but the signals conflict, such as a `RECOVERY_SIGNAL` appearing alongside a decline signal |
| `PROTECT`         | The page has strong visibility but shows early signs of volatility or a potential CTR issue                        |
| `RECOVERY_REVIEW` | `RECOVERY_SIGNAL` is present without any decline signals                                                           |
| `MONITOR`         | Other pages that appear near the top of the queue but do not meet the conditions above                             |

These actions are intended purely as **decision-support**. They give a reviewer a useful starting point and explain the signals behind each recommendation, but they do not automatically trigger a content change or publishing action.


In [24]:
# --- Reason codes, computed from the SAME eligible/test population the model scored ---
q = test_df["click_ratio_recent_prior"].quantile
IMP_Q = test_df["impression_ratio_recent_prior"].quantile

def reason_codes(row):
    codes = []
    if row["click_ratio_recent_prior"] <= q(0.25):
        codes.append("DECLINING_CLICKS")
    if row["impression_ratio_recent_prior"] <= IMP_Q(0.25):
        codes.append("DECLINING_IMPRESSIONS")
    if row["position_change"] > 0.5:
        codes.append("POSITION_WORSENING")
    if row["organic_session_change"] < 0:
        codes.append("ORGANIC_SESSION_DROP")
    if row["engagement_rate_change"] < 0:
        codes.append("ENGAGEMENT_DECLINE")
    if row["recent_impressions"] > test_df["recent_impressions"].median() and \
       row["recent_clicks"] < test_df["recent_clicks"].median():
        codes.append("HIGH_VISIBILITY_LOW_CLICKS")
    if row["click_ratio_recent_prior"] >= q(0.75):
        codes.append("RECOVERY_SIGNAL")
    if not codes:
        codes.append("STABLE_MONITOR")
    return codes

test_df["reason_codes"] = test_df.apply(reason_codes, axis=1)

def pick_action(codes):
    decline_codes = {"DECLINING_CLICKS", "POSITION_WORSENING", "DECLINING_IMPRESSIONS", "ORGANIC_SESSION_DROP"}
    has_decline  = bool(decline_codes & set(codes))
    has_recovery = "RECOVERY_SIGNAL" in codes
    if has_decline and has_recovery:
        return "INVESTIGATE"
    if has_decline:
        return "REFRESH_REVIEW"
    if has_recovery:
        return "RECOVERY_REVIEW"
    if "HIGH_VISIBILITY_LOW_CLICKS" in codes:
        return "PROTECT"
    return "MONITOR"

test_df["action"] = test_df["reason_codes"].apply(pick_action)
test_df["reason_codes_str"] = test_df["reason_codes"].apply(lambda cs: " + ".join(cs))
test_df[["content_hash_id", "model_score", "action", "reason_codes_str"]].head()


,content_hash_id,model_score,action,reason_codes_str
0,content_c9a4cf8750925d39,0.074089,REFRESH_REVIEW,DECLINING_CLICKS + HIGH_VISIBILITY_LOW_CLICKS
1,content_4ef1bcdb049fd2fc,0.008811,MONITOR,STABLE_MONITOR
2,content_13d60a6cf8b698cf,0.016602,REFRESH_REVIEW,DECLINING_CLICKS
3,content_63aef37c2e117fc2,0.006309,MONITOR,STABLE_MONITOR
4,content_48e8152390b84fa6,0.550770,RECOVERY_REVIEW,RECOVERY_SIGNAL


In [25]:
# --- Final ranked queue: Top 50 (or fewer if the eligible test population is smaller) ---
TOP_N = min(50, len(test_df))
ranked = test_df.sort_values("model_score", ascending=False).head(TOP_N).reset_index(drop=True)
ranked.insert(0, "rank", range(1, len(ranked) + 1))

RECOMMENDATION_COLS = ["rank", "content_hash_id", "model_score", "action", "reason_codes_str",
                        "recent_impressions", "recent_clicks", "click_ratio_recent_prior", "position_change"]
ranked_display = ranked[RECOMMENDATION_COLS].round(3)

ranked_display.to_csv("work/outputs/capstone_ranked_recommendations.csv", index=False)
print(f"Saved top {TOP_N} recommendations -> work/outputs/capstone_ranked_recommendations.csv")
ranked_display.head(20)

Saved top 50 recommendations -> work/outputs/capstone_ranked_recommendations.csv


,rank,content_hash_id,model_score,action,reason_codes_str,recent_impressions,recent_clicks,click_ratio_recent_prior,position_change
0,1,content_690341de91120803,0.899,REFRESH_REVIEW,DECLINING_IMPRESSIONS + POSITION_WORSENING,16007.0,72.0,1.259,0.504
1,2,content_138f7581e09c7378,0.891,MONITOR,STABLE_MONITOR,12323.0,49.0,1.351,-0.159
2,3,content_7132151ec39c08f7,0.890,MONITOR,STABLE_MONITOR,28708.0,186.0,1.889,-0.475
3,4,content_60ffc60f92ae4b26,0.889,MONITOR,ENGAGEMENT_DECLINE,80743.0,244.0,1.145,-0.380
4,5,content_bba166e1598f1453,0.886,MONITOR,STABLE_MONITOR,10104.0,63.0,1.280,0.309
5,6,content_a1ff20c72c0b697f,0.886,RECOVERY_REVIEW,RECOVERY_SIGNAL,22774.0,190.0,2.099,-0.433
6,7,content_f1466c9e20c2b57c,0.884,MONITOR,STABLE_MONITOR,8129.0,72.0,1.327,0.303
7,8,content_40baa8f1016f5742,0.883,REFRESH_REVIEW,DECLINING_IMPRESSIONS,58618.0,561.0,1.325,-0.159
8,9,content_f1eb238a6bb2d830,0.882,MONITOR,STABLE_MONITOR,5111.0,48.0,1.324,-0.018
9,10,content_15cfaf0af3f40954,0.882,MONITOR,STABLE_MONITOR,12659.0,79.0,1.250,0.153


In [26]:
print("Action distribution across the full ranked test population:")
print(test_df["action"].value_counts())
print()
print("Reason-code distribution (a row can carry more than one):")
from collections import Counter
code_counts = Counter(c for codes in test_df["reason_codes"] for c in codes)
pd.Series(code_counts).sort_values(ascending=False)

Action distribution across the full ranked test population:
action
REFRESH_REVIEW     15053
MONITOR             6526
RECOVERY_REVIEW     4483
INVESTIGATE         3744
PROTECT              496
Name: count, dtype: int64

Reason-code distribution (a row can carry more than one):


,0
POSITION_WORSENING,13569
RECOVERY_SIGNAL,8227
DECLINING_IMPRESSIONS,7576
DECLINING_CLICKS,7576
STABLE_MONITOR,6260
ORGANIC_SESSION_DROP,5039
ENGAGEMENT_DECLINE,2977
HIGH_VISIBILITY_LOW_CLICKS,1769


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [27]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os

ART_DIR = "work/artifacts/capstone"
os.makedirs(ART_DIR, exist_ok=True)

# 1. Model vs baseline metric comparison
fig, ax = plt.subplots(figsize=(6, 4))
metrics_to_plot = ["precision_at_50", "recall_at_50", "average_precision"]
x = np.arange(len(metrics_to_plot))
width = 0.35
for i, method in enumerate(results_table["method"]):
    vals = results_table.loc[results_table["method"] == method, metrics_to_plot].values.flatten()
    ax.bar(x + i * width, vals, width, label=method)
ax.set_xticks(x + width / 2)
ax.set_xticklabels(metrics_to_plot, rotation=15)
ax.axhline(y_test.mean(), color="gray", linestyle="--", linewidth=1, label="base rate")
ax.set_title("Model vs baseline (test cutoff 2026-01-31)")
ax.legend()
plt.tight_layout()
plt.savefig(f"{ART_DIR}/model_vs_baseline.png", dpi=150)
plt.close()
print("Saved model_vs_baseline.png")

Saved model_vs_baseline.png


In [28]:
# 2. Label distribution (eligible population, by split)
fig, ax = plt.subplots(figsize=(5, 4))
eligible.groupby("split")["opportunity_label"].mean().plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452"])
ax.set_ylabel("opportunity_label prevalence")
ax.set_title("Label prevalence by split")
plt.tight_layout()
plt.savefig(f"{ART_DIR}/label_distribution.png", dpi=150)
plt.close()

# 3. Feature importance
fig, ax = plt.subplots(figsize=(6, 5))
importances.sort_values().plot(kind="barh", ax=ax)
ax.set_title("Random forest feature importance")
plt.tight_layout()
plt.savefig(f"{ART_DIR}/feature_importance.png", dpi=150)
plt.close()

# 4. Score distribution
fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(model_test_score, bins=30)
ax.set_title("Model score distribution (test cutoff)")
ax.set_xlabel("predicted opportunity score")
plt.tight_layout()
plt.savefig(f"{ART_DIR}/score_distribution.png", dpi=150)
plt.close()

# 5. Top-ranked opportunity / reason-code mix
fig, ax = plt.subplots(figsize=(6, 4))
pd.Series(Counter(c for codes in ranked["reason_codes"] for c in codes)).sort_values().plot(kind="barh", ax=ax)
ax.set_title(f"Reason-code mix — top {TOP_N} ranked pages")
plt.tight_layout()
plt.savefig(f"{ART_DIR}/top_reason_codes.png", dpi=150)
plt.close()

print("All charts saved under", ART_DIR)
os.listdir(ART_DIR)

All charts saved under work/artifacts/capstone


['model_vs_baseline.png',
 'top_reason_codes.png',
 'score_distribution.png',
 'label_distribution.png',
 'feature_importance.png']

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.


## ML-12 — Closing

### 5-Minute Demo

* **Problem:** Help SEO reviewers decide which content pages to review first.
* **Data:** 79M-row FlyRank warehouse queried with DuckDB over `hf://`.
* **Method:** Leakage-safe historical features, a data-driven future-decline label, and time-aware validation.
* **Results:** Random Forest vs. a transparent baseline. Top-50 precision was close to chance, but Average Precision was about **2× higher** than the baseline.
* **Output:** Ranked pages with reason codes and actions such as `REFRESH_REVIEW`, `MONITOR`, and `PROTECT`.
* **Takeaway:** The model provides useful ranking signals, but it is **decision-support, not a guarantee**.

### Employer Summary

Built a leakage-safe ML ranking system on a **79M-row search dataset** using DuckDB. Compared a Random Forest with a transparent baseline using time-aware validation and reported the results honestly: top-50 performance was mixed, while overall ranking quality was roughly **2× better** than the baseline.
